In [1]:
import pandas as pd
import json

train_with_news = pd.read_parquet("../data/processed/train_with_news.parquet")
dev_with_news = pd.read_parquet("../data/processed/dev_with_news.parquet")

with open("../data/processed/user_idx_map.json") as f:
    user_idx_map = json.load(f)
with open("../data/processed/item_idx_map.json") as f:
    item_idx_map = json.load(f)

In [3]:
from scipy.sparse import csr_matrix
import numpy as np

train_rows = train_with_news["user_id"].map(user_idx_map)
train_cols = train_with_news["item_id"].map(item_idx_map)

valid = train_rows.notna() & train_cols.notna()
train_rows = train_rows[valid].astype(int)
train_cols = train_cols[valid].astype(int)

data = np.ones(len(train_rows), dtype=np.float32)
train_matrix = csr_matrix(
    (data, (train_rows, train_cols)),
    shape=(len(user_idx_map), len(item_idx_map))
)
train_matrix.data[:] = 1  # collapse repeated (user, item) pairs to a single positive entry
train_matrix.eliminate_zeros()

In [4]:
dev_user_known = dev_with_news["user_id"].isin(user_idx_map)
dev_item_known = dev_with_news["item_id"].isin(item_idx_map)

cold_start_users = (~dev_user_known).sum()
cold_start_items = (~dev_item_known).sum()

print("Cold-start dev rows (unknown user):", cold_start_users)
print("Cold-start dev rows (unknown item):", cold_start_items)

Cold-start dev rows (unknown user): 2151528
Cold-start dev rows (unknown item): 83734


In [5]:
dev_warm = dev_with_news[dev_user_known & dev_item_known].copy()
print("Warm-start dev rows:", len(dev_warm))

Warm-start dev rows: 315614


In [6]:
train_pairs = set(zip(train_rows, train_cols))

dev_rows = dev_warm["user_id"].map(user_idx_map).astype(int)
dev_cols = dev_warm["item_id"].map(item_idx_map).astype(int)

train_seen_mask = [
    (u, i) in train_pairs for u, i in zip(dev_rows, dev_cols)
]
train_seen_count = sum(train_seen_mask)
print("Train-seen pairs removed from dev:", train_seen_count)

keep_mask = [not seen for seen in train_seen_mask]
dev_rows_final = dev_rows[keep_mask]
dev_cols_final = dev_cols[keep_mask]

Train-seen pairs removed from dev: 308814


In [7]:
dev_data = np.ones(len(dev_rows_final), dtype=np.float32)
dev_matrix = csr_matrix(
    (dev_data, (dev_rows_final, dev_cols_final)),
    shape=(len(user_idx_map), len(item_idx_map))
)
dev_matrix.eliminate_zeros()

In [8]:
from scipy.sparse import save_npz

save_npz("../data/processed/train_interactions.npz", train_matrix)
save_npz("../data/processed/dev_interactions.npz", dev_matrix)

In [9]:
print("Train matrix shape:", train_matrix.shape)
print("Train matrix nnz:", train_matrix.nnz)
print("Train density:", train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1]))

print("Dev matrix shape:", dev_matrix.shape)
print("Dev matrix nnz:", dev_matrix.nnz)

Train matrix shape: (50000, 39865)
Train matrix nnz: 1148447
Train density: 0.0005761680672268908
Dev matrix shape: (50000, 39865)
Dev matrix nnz: 6746
